<a href="https://colab.research.google.com/github/saurav-singh321/Flask_ML/blob/main/spaceship%20titanic.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:
#Importing necessary Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import ExtraTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score,RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import warnings

warnings.filterwarnings('ignore')

In [ ]:
#Loading train and test data
train = pd.read_csv('https://raw.githubusercontent.com/saurav-singh321/Flask_ML/main/train_spaceship.csv')
test = pd.read_csv('https://raw.githubusercontent.com/saurav-singh321/Flask_ML/main/test_spaceship.csv')
train.head()

Train - Personal records for about two-thirds (~8700) of the passengers, to be used as training data.

`PassengerId` - A unique Id for each passenger. Each Id takes the form gggg_pp where gggg indicates a group the passenger is travelling with and pp is their number within the group. People in a group are often family members, but not always.

`HomePlanet` - The planet the passenger departed from, typically their planet of permanent residence.
    
`CryoSleep` - Indicates whether the passenger elected to be put into suspended animation for the duration of the voyage. Passengers in cryosleep are confined to their cabins.

`Cabin`- The cabin number where the passenger is staying. Takes the form deck/num/side, where side can be either P for Port or S for Starboard.
    
`Destination` - The planet the passenger will be debarking to.
    
`Age` - The age of the passenger.
    
`VIP` - Whether the passenger has paid for special VIP service during the voyage.
    
`RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck` - Amount the passenger has billed at each of the Spaceship Titanic's many luxury amenities.
    
`Name` - The first and last names of the passenger.
    
`Transported` - Whether the passenger was transported to another dimension. This is the target, the column you are trying to predict.

test.csv - Personal records for the remaining one-third (~4300) of the passengers, to be used as test data. Your task is to predict the value of Transported for the passengers in this set. 

### Checking null values in train and test

In [ ]:
combine = [train,test]
for df in combine:
    print(df.isna().sum())
    print()

# Features with nan

Here, we are capturing the names of the feature from train and test  which has nan values and also how many nan values

In [ ]:
# features with na
na_features_train = [features for features in train.columns if (train[features].isna().sum()>0)]
na_features_test = [features for features in test.columns if (test[features].isna().sum()>0)]

print(na_features_train)
print(na_features_test)
print()


#count of features with na
na_features_train_count = {features:train[features].isna().sum() for features in na_features_train}
na_features_test_count = {features:test[features].isna().sum() for features in na_features_test}

print(na_features_train_count)
print(na_features_test_count)

### Checking if the data is balanced or unbalanced

In [ ]:
sns.countplot(x = 'Transported',data = train)
train['Transported'].value_counts()

The data is balanced.

# Numerical features

In [ ]:
df = train.copy()
df1 = test.copy()

numerical_features_train = df.select_dtypes(exclude='O').columns
numerical_features_test = df1.select_dtypes(exclude='O').columns

print(numerical_features_train)
print(numerical_features_test)

## Discrete numerical features

In [ ]:
df = train.copy()
df1 = test.copy()

discrete_numerical_features_train = [features for features in numerical_features_train if len(df[features].unique())<20]
discrete_numerical_features_test = [features for features in numerical_features_test if len(df1[features].unique())<20]

print(discrete_numerical_features_train)
print(discrete_numerical_features_test)

We have no discrete features in our dataset. The transported feature is target feature

## Continuous numerical features

In [ ]:
continuous_numerical_features_train= [features for features in numerical_features_train if features not in discrete_numerical_features_train]
continuous_numerical_features_test = [features for features in numerical_features_test  if features not in discrete_numerical_features_test]

print(continuous_numerical_features_train)
print(continuous_numerical_features_test)

In [ ]:
df = train.copy()
df1 = test.copy()

#distribution of continuous data
for features in continuous_numerical_features_train:
    df[features].hist(bins = 25)
    plt.xlabel(features)
    plt.ylabel('Number of people')
    plt.show()
    

## Observation - 
1. Most of the people are from 18-22 `Age` i.e adults.

## Outlier

In [ ]:
df = train.copy()
# Checking outlier
for features in continuous_numerical_features_train:
    df.boxplot(column=features)
    plt.title(features)
    plt.show()

We have outliers in continuous features which will be removed later.

# Categorical features

In [ ]:
df = train.copy()
df1 = test.copy()

categorical_features_train = df.select_dtypes(include='O').columns
categorical_features_test = df1.select_dtypes(include='O').columns

print(categorical_features_train)
print(categorical_features_test)

## Cardinality of categorical features

In [ ]:
categories_count_train = {features:len(df[features].unique()) for features in categorical_features_train}
categories_count_train = dict(sorted(categories_count_train.items(), key = lambda ele: ele[1]))
print(categories_count_train)


categories_count_test = {features:len(df1[features].unique()) for features in categorical_features_test}
categories_count_test = dict(sorted(categories_count_test.items(), key = lambda ele: ele[1]))
print(categories_count_test)

In [ ]:
# Relation of categorical features with target variable
for features in categorical_features_train:
    if features not in 'Cabin'+'Name'+'PassengerId':
        df.groupby(features)['Transported'].sum().plot(kind='bar',color=['cornflowerblue','darkorange','darkkhaki'])
        plt.show()

### Observation - 

1. Most of the passenger are from `Earth`.
2. Most of the passenger were in `cyrosleep`.
3. Most of the passenger's destination is `Trappist-1e`.
4. Most of the people were not opt for `VIP` service.

# Correlation

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(train.corr(),annot=True,vmin=-1,vmax=1,cmap='viridis')
plt.show()

### Observation
We can assume our features to be independent because no two features are highly correlated.

# Feature Engineering

Converting Output feature to int

In [ ]:
train['Transported'] = train['Transported'].astype(int)
train['Transported']

### Making Passenger Id as Index

We are doing this because when we again split the data in train and test then these index help us to categorize the data.

In [ ]:
train = train.set_index('PassengerId')
test = test.set_index('PassengerId')

### Combining Train and test

In [ ]:
df = pd.concat([train,test],axis=0)
df

In [ ]:
df.describe().T

### Detecting and Removing Outliers

In [ ]:
def Remove_Outlier_Indices(df):
    Q1 = df.quantile(0.25)
    Q3 = df.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound=Q1 - 1.5 * IQR
    upper_bound=Q3 + 1.5 * IQR
    trueList = ~((df < lower_bound) |(df > upper_bound))
    return trueList

In [ ]:
nonOutlierList = Remove_Outlier_Indices(df)
# Non-Outlier Subset of the Given Dataset
dfSubset = df[nonOutlierList]
dfSubset

## Handling missing values 

In [ ]:
df.isnull().sum()

There are so many missing values in every feature. 

In [ ]:
dfSubset['HomePlanet']  = dfSubset['HomePlanet'].fillna(dfSubset['HomePlanet'].mode().item())
dfSubset['CryoSleep']   = dfSubset['CryoSleep'].fillna(dfSubset['CryoSleep'].mode().item())
dfSubset['Cabin']       = dfSubset['Cabin'].fillna('X/1900/X') # becauSe max iS 1894
dfSubset['Destination'] = dfSubset['Destination'].fillna(dfSubset['Destination'].mode().item())
dfSubset['VIP']         = dfSubset['VIP'].fillna(dfSubset['VIP'].mode().item())
dfSubset['Age']         = dfSubset['Age'].fillna(dfSubset['Age'].mean())

### Making Age group

In [ ]:
def age_group(feature):
    group = ''
    if feature<5.0:
        group = 'Baby'
    elif feature>=5.0 and feature<18.0:
        group = 'children'
    elif feature>=18.0 and feature<65.0:
        group = 'adult'
    elif feature>=65.0:
        group = 'senior'
    else:
        group ='unknown'
        
    return group

In [ ]:
# df['AgeGroup'] = df['Age'].map(age_group)
dfSubset['AgeGroup'] = dfSubset['Age'].map(age_group)

## Now filling the expenses wrt to age group and creating a new col of total expense

In [ ]:
cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
for i in cols:
    print(i)
#     a = df.groupby('AgeGroup')[i].mean()
    a = dfSubset.groupby('AgeGroup')[i].mean()
    print(a)
    print()

In [ ]:
conditions  = [dfSubset['AgeGroup'] == 'Baby',dfSubset['AgeGroup'] == 'adult',dfSubset['AgeGroup'] == 'children',dfSubset['AgeGroup'] == 'senior']

choices_room = [0.0,5.8,3.6,6.8]
choices_food = [0.0,9.4,4.6,4.9]
choices_shop = [0.0,3.6,2.0,3.7]
choices_spa  = [0.0,8.4,3.2,16.0]
choices_vr   = [0.0,5.7,2.3,2.2]

    
dfSubset['RoomService']  = np.select(conditions, choices_room, default=np.nan)
dfSubset['FoodCourt']    = np.select(conditions, choices_food, default=np.nan)
dfSubset['ShoppingMall'] = np.select(conditions, choices_shop, default=np.nan)
dfSubset['Spa']          = np.select(conditions, choices_spa, default=np.nan)
dfSubset['VRDeck']       = np.select(conditions, choices_vr, default=np.nan)

In [ ]:
dfSubset.isnull().sum()

Note - We don't have any null values. The `transported` values are from test data.

### Creating a new column of total expenses

In [ ]:
dfSubset['Total_Expense'] = dfSubset['RoomService']+dfSubset['FoodCourt']+dfSubset['Spa']+dfSubset['VRDeck']
dfSubset

Data provided the segregation of Cabin. So, diving the Cabin as described in the data

In [ ]:
dfSubset['Cabin_deck']   = dfSubset['Cabin'].apply(lambda x: x.split('/')[0])
dfSubset['Cabin_number'] = dfSubset['Cabin'].apply(lambda x: x.split('/')[1]).astype(int)
dfSubset['Cabin_side']   = dfSubset['Cabin'].apply(lambda x: x.split('/')[2])
dfSubset.drop(['Name','Cabin'],axis=1,inplace=True) #because no use
dfSubset

In [ ]:
dfSubset.corr()

# One hot encoding

One hot encoding will create multiple columns. So, what I'm doing here is converting the categories as `other` which are occuring very less number of times

In [ ]:
dfSubset['Cabin_deck'].replace({'A':'O',
                               'D':'O',
                               'T':'O',
                               'X':'O'},inplace=True)

Doing One hot encoding of nominal and label encoding of ordinal

In [ ]:
new_cat = ['HomePlanet','CryoSleep','Destination','VIP','Cabin_deck','Cabin_side']
for features in new_cat:
    dummy = pd.get_dummies(dfSubset[features],drop_first=True)
    dfSubset = pd.concat([dfSubset,dummy],axis=1)
    dfSubset.drop(features,axis=1,inplace=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
dfSubset['AgeGroup'] = le.fit_transform(dfSubset["AgeGroup"])
print(features, le.classes_)

In [ ]:
dfSubset

### Splitting the data again in training and testing

As mentioned earlier that the passenger id will help us to split the data.

In [ ]:
train = dfSubset[dfSubset.index.isin(train.index)] # For training + validation
test = dfSubset[dfSubset.index.isin(test.index)] # For test / prediction

In [ ]:
train.shape

Dropping multiple features along with target feature as they are of no use.

In [ ]:
X = train.drop(columns=['Transported','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck'])
y = train['Transported']
X_test = test.drop(columns=['Transported','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck'])

In [ ]:
X_train,X_valid,y_train,y_valid = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
#Scaling the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

In [ ]:
tree_reg = ExtraTreeClassifier()
tree_reg.fit(X_train_scaled,y_train)
print(tree_reg.feature_importances_)

#Top 5 important feature
features_importance = pd.Series(tree_reg.feature_importances_,index=X.columns)
features_importance.nlargest(5).plot(kind='barh')
plt.show()

These are the top 5 features which are important

In [ ]:
from sklearn.svm import SVC
model = SVC()

model.fit(X_train_scaled,y_train)
y_pred = model.predict(X_valid_scaled)

In [ ]:
accuracy_score(y_valid,y_pred)

In [ ]:
# X_test
sub = model.predict(X_test)
sub

In [ ]:
import pickle
pickle.dump(model,open('model1.pkl','wb'))

In [ ]:
X

In [ ]:
X['AgeGroup'].unique()

In [ ]:
adult 1 children 2